In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.visualization.mp_renderer import MPRenderer

from source.crmonitor.common.helper import load_yaml
from source.crmonitor.common.world import World
from source.crmonitor.evaluation.evaluation import RuleEvaluator
from source.crmonitor.evaluation.visualization import plot_rule_visualization

# add the crmonitor folder to python path

# —— Project root directory ——
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# —— Load configuration ——
cfg = yaml.safe_load((project_root / "configurations" / "scenario.yaml").read_text())
bus_stop = cfg["scenario"]["type"]

# —— Load scenario file ——
scenario_file = project_root / "scenarios" / bus_stop / f"{bus_stop}.xml"
scenario, planning_problem_set = CommonRoadFileReader(scenario_file).open(
    lanelet_assignment=True
)
# —— Get the first dynamic obstacle (ego vehicle) ——
ego_vehicle = next(iter(scenario.dynamic_obstacles))



In [ ]:
# for i in range(0, 100):
#     plt.figure(figsize=(20,30))
#     rnd = MPRenderer()
#     rnd.draw_params.time_begin = i
#     #rnd.draw_params.dynamic_obstacle.show_label = True
#     rnd.draw_params.dynamic_obstacle.draw_direction = True
#     rnd.draw_params.dynamic_obstacle.vehicle_shape.occupancy.shape.facecolor = 'yellow'
#     rnd.draw_params.static_obstacle.occupancy.draw_occupancies = True
#     rnd.draw_params.traffic_sign.show_label = True
#     rnd.draw_params.traffic_sign.draw_traffic_signs = True
#     rnd.draw_params.lanelet_network.traffic_sign.draw_traffic_signs = True
#     scenario.draw(rnd)
#     ego_vehicle.draw(rnd)
#     rnd.render()
#     plt.show()

In [ ]:
config_path = "../source/crmonitor/config_pt.yaml"
config = load_yaml(str(config_path))
rules_path = "../source/crmonitor/traffic_rules_pt.yaml"
traffic_rules = load_yaml(str(rules_path))

world = World.create_from_scenario(scenario, config=config)
world_ego_vehicle = next(iter(world.vehicles))
rule_set = ["RB_2","RB_1"]
rule_evaluator_list = list()

for rule in rule_set:
    rule_evaluator_list.append(RuleEvaluator.create_from_config(world, world_ego_vehicle, rule=rule, traffic_rules_config=traffic_rules))

In [ ]:
visualization_config = {
    'keeps_lane_speed_limit_with_minmax': {
        'ego': [300] # we want to visualize the cut-in predicate always for the vehicle-pair (1004, 1000),
        # even if the predicate instance is not effective.
    }
}


# For a more rectangular scenario:
scenario_fig_size=(25., 3.)
scenario_scale_compared_to_other_plots=6
# For evaluating all rules together
flag_rule_conjunction=True

current_time_step = 100
while current_time_step < 200:
    plot_rule_visualization(scenario,
                            world_ego_vehicle.id,
                            current_time_step,
                            rule_evaluator_list,
                            visualization_config=visualization_config,
                            scenario_fig_size=scenario_fig_size,
                            bar_chart_plot_limits=(-1.0, 1.0),
                            rule_robustness_course_plot_limits=(-1.0, 1.0),
                            flag_plot_predicate_bar_chart=True,
                            flat_plot_rule_robustness_course=True,
                            scenario_plot_limits=None,
                            flag_rule_conjunction=flag_rule_conjunction
                            )
    current_time_step += 1
    plt.show()